# Environment Check

Run this first on any new machine. It confirms the GPU is actually being used and that the dataset config resolves before you start a long training run.

In [1]:
import torch

print("torch     ", torch.__version__)
print("cuda build", torch.version.cuda)
print("available ", torch.cuda.is_available())
print("device    ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

torch      2.5.1+cu121
cuda build 12.1
available  True
device     NVIDIA GeForce GTX 1060


In [3]:
import ultralytics

# YOLO26 checkpoints need ultralytics >= 8.4.60
print("ultralytics", ultralytics.__version__)
ultralytics.checks()

Ultralytics 8.4.133  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1060, 6144MiB)
Setup complete  (8 CPUs, 15.9 GB RAM, 232.7/930.9 GB disk)


## Dataset config

Confirm the class list here matches the `data.yaml` shipped in the Roboflow export. A mismatch trains correct-looking weights that predict the wrong labels.

In [7]:
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd().parent

data_cfg = yaml.safe_load((REPO_ROOT / "data" / "css-data.yaml").read_text())
vocab = yaml.safe_load((REPO_ROOT / "data" / "vocabulary.yaml").read_text())

print("classes:")
for i, name in data_cfg["names"].items():
    print(f"  {i:>2}  {name}")

print("\nsubject: ", vocab["subject"])
print("ppe:     ", list(vocab["ppe"]))
print("negative:", list(vocab["negative"]))
print("ignore:  ", vocab["ignore"])

classes:
   0  Excavator
   1  Gloves
   2  Hardhat
   3  Ladder
   4  Mask
   5  NO-Hardhat
   6  NO-Mask
   7  NO-Safety Vest
   8  Person
   9  SUV
  10  Safety Cone
  11  Safety Vest
  12  bus
  13  dump truck
  14  fire hydrant
  15  machinery
  16  mini-van
  17  sedan
  18  semi
  19  trailer
  20  truck
  21  truck and trailer
  22  van
  23  vehicle
  24  wheel loader

subject:  Person
ppe:      ['Hardhat', 'Safety Vest', 'Mask', 'Gloves']
negative: ['NO-Hardhat', 'NO-Safety Vest', 'NO-Mask']
ignore:   ['Excavator', 'Ladder', 'SUV', 'Safety Cone', 'bus', 'dump truck', 'fire hydrant', 'machinery', 'mini-van', 'sedan', 'semi', 'trailer', 'truck', 'truck and trailer', 'van', 'vehicle', 'wheel loader']


In [8]:
# Every vocabulary name must exist in the dataset config
declared = set(data_cfg["names"].values())
referenced = {vocab["subject"], *vocab["ppe"], *vocab["negative"], *vocab["ignore"]}

missing = referenced - declared
unmapped = declared - referenced

print("missing from dataset :", missing or "none")
print("unmapped in vocabulary:", unmapped or "none")
assert not missing, f"vocabulary references classes the dataset does not define: {missing}"

missing from dataset : none
unmapped in vocabulary: none
